# Office OS — GRPO Fine-Tuning with Unsloth + TRL

**Multi-agent startup simulation where 6 AI agents collaborate to grow a SaaS company.**

This notebook demonstrates the full training pipeline:
1. **Install** — Unsloth, TRL, openenv-core, vLLM deps
2. **Generate** — Expert training data from the real Office OS simulator
3. **Train** — GRPO with dual reward functions (format scorer + LLM judge)
4. **Inference** — Run fine-tuned agents vs base model side-by-side
5. **Hot-load** — Push LoRA into running vLLM server

> Runs on **Colab T4** (7B model) or **A100** (14B model). ~15 min per role.

[![GitHub](https://img.shields.io/badge/GitHub-SuperOffice__env-blue)](https://github.com/bvsbharat/SuperOffice_env)

## 1. Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install "trl>=0.12" "datasets>=3.0" "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.44"
!pip install "openenv-core[core]>=0.2.0"
!pip install wandb huggingface_hub python-dotenv anthropic boto3 gspread httpx

# Clone the Office OS repo
!git clone https://github.com/bvsbharat/SuperOffice_env.git 2>/dev/null || (cd SuperOffice_env && git pull)

import sys, os
sys.path.insert(0, os.path.abspath("SuperOffice_env/office_os"))
os.environ.pop("GOOGLE_SHEETS_CREDENTIALS", None)
os.environ.pop("GOOGLE_SHEETS_SPREADSHEET_ID", None)
print("Dependencies installed.")

In [ ]:
# ── Configuration ──────────────────────────────────────────────
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"   # 7B for T4, 14B for A100
TRAIN_ROLE = "dev"                          # Which role to train (try: dev, sales, ceo)
NUM_EPISODES = 10                           # More = better data (50+ recommended)
DAYS_PER_EPISODE = 30
LEARNING_RATE = 2e-5
MAX_STEPS = 50

# Optional: W&B and HuggingFace
WANDB_PROJECT = ""   # e.g. "office-os"
HF_REPO = ""         # e.g. "username/office-os-loras"

# Uncomment to enable:
# os.environ["WANDB_API_KEY"] = "your-key"
# os.environ["HF_TOKEN"] = "your-token"

## 2. Generate Training Data from the Real Simulator

We run the full Office OS environment with expert rule-based policies across 5 scenarios:
**baseline**, **competitor launch**, **series A pressure**, **churn spike**, **viral moment**.

Each turn produces a trajectory: the observation the agent sees + the expert's optimal action.

In [ ]:
import json, random, logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("training")

from agents.prompts import ROLE_PROMPTS
from agents.llm_agent import LLMAgent
from market.config import AGENT_ROLES, ROLE_ACTIONS, TURNS_PER_DAY, EPISODE_DAYS
from server.office_os_environment import OfficeOsEnvironment
from models import OfficeOsAction

print(f"Roles: {AGENT_ROLES}")
print(f"Actions for {TRAIN_ROLE}: {ROLE_ACTIONS[TRAIN_ROLE]}")
print(f"Turns per day: {TURNS_PER_DAY}, Days per episode: {DAYS_PER_EPISODE}")
print(f"Scenarios: baseline, competitor, series_a, churn, viral")

In [ ]:
# ── Expert policies — optimal decisions per role ─────────────
# Hand-crafted to make the best move given game state.
# Reasoning references actual KPIs, customer names, features.

def expert_ceo(obs, turn):
    kpis = obs.get("kpis", {})
    day, rev, pipeline = obs.get("day", 1), kpis.get("total_revenue", 0), kpis.get("pipeline_value", 0)
    features, budget = kpis.get("features_shipped", 0), kpis.get("budget_remaining", 0)
    if day <= 2:
        return {"action_type": "SET_OKRS", "target": "Q1_Revenue_Growth",
                "parameters": {"key_results": ["Ship 2 features", "Close 3 deals", "10 content pieces"]},
                "reasoning": f"Day {day}: Setting OKRs. Pipeline ${pipeline:,.0f}, {features} features shipped.",
                "message": "sales: top priority is closing deals. dev: ship features ASAP."}
    if budget < 5000:
        return {"action_type": "SEND_DIRECTIVE", "target": "Budget conservation", "parameters": {},
                "reasoning": f"Budget critically low at ${budget:,.0f}. Focus on converting pipeline (${pipeline:,.0f}).",
                "message": f"marketing: STOP paid campaigns. sales: close existing pipeline."}
    if features == 0 and day > 5:
        return {"action_type": "SEND_DIRECTIVE", "target": "Ship features urgently", "parameters": {},
                "reasoning": f"Day {day} with 0 features — sales has nothing to demo. Pipeline ${pipeline:,.0f}.",
                "message": "dev: ship ASAP. hr: clear all blockers."}
    if rev == 0 and day > 10:
        return {"action_type": "REVIEW_STRATEGY", "target": "Revenue review", "parameters": {},
                "reasoning": f"Day {day} with $0 revenue. Pipeline ${pipeline:,.0f}, {features} features.",
                "message": "sales: what's blocking closure? dev: are customer features shipped?"}
    return random.choice([
        {"action_type": "SEND_DIRECTIVE", "target": "Keep pushing", "parameters": {},
         "reasoning": f"Day {day}: Rev=${rev:,.0f}, Pipeline=${pipeline:,.0f}. Keeping team focused.",
         "message": "sales: advance pipeline. dev: keep building."},
        {"action_type": "REVIEW_STRATEGY", "target": f"Day {day} review", "parameters": {},
         "reasoning": f"Check: Rev=${rev:,.0f}, Pipeline=${pipeline:,.0f}, Features={features}.",
         "message": f"team: rev=${rev:,.0f}, pipeline=${pipeline:,.0f}. {'On track.' if rev > 0 else 'Accelerate.'}"},
    ])

def expert_dev(obs, turn):
    rd = obs.get("role_data", {})
    in_progress, backlog, bugs = rd.get("features_in_progress", []), rd.get("backlog", []), rd.get("bug_reports", [])
    shipped = rd.get("team_status", {}).get("dev", {}).get("shipped", [])
    stability = obs.get("kpis", {}).get("product_stability", 1.0)
    ready = [f for f in in_progress if f.get("turns_remaining", 99) <= 0]
    if ready:
        return {"action_type": "SHIP_RELEASE", "target": ready[0]["name"], "parameters": {},
                "reasoning": f"'{ready[0]['name']}' complete — feature #{len(shipped)+1}. Shipping to unblock sales.",
                "message": f"sales: shipped {ready[0]['name']}! Ready for demos. content: write case study."}
    if in_progress:
        b = in_progress[0]
        return {"action_type": "BUILD_FEATURE", "target": b["name"], "parameters": {},
                "reasoning": f"Continuing '{b['name']}' — {b.get('turns_remaining','?')} turns left.",
                "message": f"sales: building {b['name']}, {b.get('turns_remaining','?')} turns left."}
    critical = [b for b in bugs if b.get("severity") in ("critical", "high")]
    if critical:
        bug = critical[0]
        return {"action_type": "FIX_BUG", "target": bug.get("name", "bug"), "parameters": {},
                "reasoning": f"Fixing critical bug '{bug.get('name','')}'. Stability {stability:.0%}.",
                "message": "sales: fixing bug. customer: fix incoming."}
    if backlog:
        item = backlog[0]
        return {"action_type": "BUILD_FEATURE", "target": item["name"], "parameters": {},
                "reasoning": f"Starting '{item['name']}' from backlog. {len(shipped)} features shipped.",
                "message": f"sales: starting {item['name']}. ~3 turns."}
    return {"action_type": "REFACTOR", "target": "codebase", "parameters": {},
            "reasoning": f"Nothing pending. Stability {stability:.0%}, refactoring for +5%.",
            "message": "sales: improving stability via refactor."}

def expert_marketing(obs, turn):
    kpis = obs.get("kpis", {})
    budget, pipeline = kpis.get("budget_remaining", 0), kpis.get("pipeline_value", 0)
    traffic, conv = kpis.get("website_traffic", 0), kpis.get("conversion_rate", 0)
    if budget < 3000:
        return {"action_type": "OPTIMIZE_FUNNEL", "target": "conversion", "parameters": {},
                "reasoning": f"Budget low (${budget:,.0f}). Conversion {conv:.1%}. Free optimization.",
                "message": "sales: optimizing funnel. ceo: conserving budget."}
    if pipeline < 20000:
        topic = random.choice(["Enterprise security", "SaaS professionals", "Digital transformation"])
        return {"action_type": "LAUNCH_CAMPAIGN", "target": topic, "parameters": {},
                "reasoning": f"Pipeline thin (${pipeline:,.0f}). Launching campaign. Budget ${budget:,.0f}.",
                "message": f"sales: launched '{topic}' campaign, expect leads."}
    if conv < 0.03:
        return {"action_type": "A_B_TEST", "target": "landing page CTA", "parameters": {},
                "reasoning": f"Conversion {conv:.1%} below 3% target. A/B test for permanent boost.",
                "message": f"sales: A/B testing to improve {conv:.1%} conversion."}
    return {"action_type": "RUN_AD", "target": random.choice(["enterprise", "startup", "SMB"]) + " segment",
            "parameters": {}, "reasoning": f"Maintaining lead flow. Pipeline ${pipeline:,.0f}, traffic {traffic:,.0f}.",
            "message": "sales: ads running, qualify incoming leads."}

def expert_sales(obs, turn):
    pipeline = obs.get("role_data", {}).get("pipeline", [])
    stage_to_action = {"lead": "QUALIFY_LEAD", "qualified": "RUN_DEMO", "demo": "SEND_PROPOSAL",
                       "proposal": "CLOSE_DEAL", "negotiation": "CLOSE_DEAL"}
    priority = {"proposal": 0, "negotiation": 1, "demo": 2, "qualified": 3, "lead": 4}
    active = [c for c in pipeline if c.get("stage") in stage_to_action]
    if not active:
        return {"action_type": "UPDATE_SHEET", "target": "pipeline", "parameters": {},
                "reasoning": "Pipeline empty. Syncing sheet. Need marketing campaigns.",
                "message": "marketing: pipeline EMPTY, need campaigns."}
    stale = [c for c in active if c.get("days_since_contact", 0) > 3]
    if stale:
        c = stale[0]
        return {"action_type": "FOLLOW_UP", "target": c["name"], "parameters": {},
                "reasoning": f"{c['name']} stale ({c.get('days_since_contact',0)}d). Preventing decay.",
                "message": f"dev: {c['name']} needs attention."}
    active.sort(key=lambda c: priority.get(c["stage"], 99))
    c = active[0]
    action = stage_to_action[c["stage"]]
    params = {}
    if action == "CLOSE_DEAL":
        tier = {"startup": "monthly", "smb": "6_month", "enterprise": "annual"}.get(c.get("company_size"), "monthly")
        params = {"contract_tier": tier}
    return {"action_type": action, "target": c["name"], "parameters": params,
            "reasoning": f"Advancing {c['name']} from {c['stage']} (${c.get('budget',0):,.0f}). Pain: {c.get('pain_point','')}",
            "message": f"dev: working {c['name']}. content: need materials."}

def expert_content(obs, turn):
    shipped = obs.get("role_data", {}).get("team_status", {}).get("dev", {}).get("shipped", [])
    shipped_names = [f if isinstance(f, str) else f.get("name", "") for f in shipped]
    traffic = obs.get("kpis", {}).get("website_traffic", 0)
    if shipped_names:
        feat = shipped_names[0]
        return {"action_type": "WRITE_CASE_STUDY", "target": f"{feat} success story",
                "parameters": {"feature": feat},
                "reasoning": f"Case study on shipped '{feat}' — highest-converting content for Sales.",
                "message": f"sales: case study on {feat} coming. marketing: will share for campaigns."}
    topic = random.choice(["SaaS Security Guide", "Onboarding Best Practices", "API Integration Patterns"])
    return {"action_type": "WRITE_BLOG", "target": topic, "parameters": {"topic": topic},
            "reasoning": f"No shipped features yet. Blog to drive traffic ({traffic:,.0f}).",
            "message": f"marketing: blog on '{topic}' coming. Please amplify."}

def expert_hr(obs, turn):
    kpis = obs.get("kpis", {})
    blockers, velocity = kpis.get("blockers", 0), kpis.get("team_velocity", 1.0)
    budget, day = kpis.get("budget_remaining", 0), obs.get("day", 1)
    if blockers > 0:
        return {"action_type": "RESOLVE_BLOCKER", "target": "team blocker", "parameters": {},
                "reasoning": f"{blockers} blockers dragging velocity to {velocity:.1f}x.",
                "message": "dev: clearing blockers. ceo: fixing velocity."}
    if velocity < 0.8 and budget > 5000:
        return {"action_type": "HIRE_CONTRACTOR", "target": "developer", "parameters": {},
                "reasoning": f"Velocity {velocity:.1f}x critically low. Hiring to boost.",
                "message": "dev: contractor incoming. ceo: investing in velocity."}
    if day % 3 == 1:
        return {"action_type": "TRACK_OKRS", "target": "quarterly objectives", "parameters": {},
                "reasoning": f"Day {day} OKR check. Velocity {velocity:.1f}x.",
                "message": "ceo: OKR progress update."}
    focus = random.choice(["feature development", "deal closure", "stability"])
    return {"action_type": "PLAN_SPRINT", "target": focus, "parameters": {},
            "reasoning": f"Sprint focused on {focus}. Velocity {velocity:.1f}x.",
            "message": f"dev: sprint on {focus}. team: execute."}

def expert_customer(obs, turn):
    kpis = obs.get("kpis", {})
    nps, stab, feat = kpis.get("nps_score", 50), kpis.get("product_stability", 1.0), kpis.get("features_shipped", 0)
    if obs.get("day", 1) % 5 == 1:
        return {"action_type": "EVALUATE_PRODUCT", "target": "assessment", "parameters": {},
                "reasoning": f"NPS={nps:.0f}, stability={stab:.0%}, features={feat}.",
                "message": f"dev: {'good progress' if feat > 0 else 'waiting for features'}."}
    if nps > 40 and stab > 0.7 and feat > 1:
        return {"action_type": "REFER_LEAD", "target": "colleague", "parameters": {},
                "reasoning": f"Satisfied: NPS={nps:.0f}, {feat} features. Referring.",
                "message": "sales: referring a colleague."}
    if feat > 0 and kpis.get("customer_satisfaction", 0) > 0.4:
        return {"action_type": "RENEW_CONTRACT", "target": "contract", "parameters": {},
                "reasoning": f"Product improving: {feat} features, NPS={nps:.0f}. Renewing.",
                "message": "sales: renewing. Keep shipping."}
    feat_name, desc = random.choice([
        ("Mobile App", "Field teams need mobile access"),
        ("Webhooks", "Need real-time integrations"),
        ("Custom Reports", "Executive reporting requirement"),
    ])
    return {"action_type": "REQUEST_FEATURE", "target": feat_name, "parameters": {"description": desc},
            "reasoning": f"Requesting {feat_name}: {desc}.", "message": f"dev: need {feat_name}."}

EXPERT_POLICIES = {
    "ceo": expert_ceo, "dev": expert_dev, "marketing": expert_marketing,
    "sales": expert_sales, "content": expert_content, "hr": expert_hr, "customer": expert_customer,
}
print(f"Expert policies defined for: {list(EXPERT_POLICIES.keys())}")

In [ ]:
# ── Run the REAL simulator with expert policies ─────────────────
SCENARIOS = ["baseline", "competitor", "series_a", "churn", "viral"]
all_records = []

for ep in range(1, NUM_EPISODES + 1):
    scenario = SCENARIOS[(ep - 1) % len(SCENARIOS)]
    env = OfficeOsEnvironment(scenario=scenario)
    obs = env.reset()
    agents = {role: LLMAgent(role=role) for role in AGENT_ROLES}

    turn, role_idx = 0, 0
    while not obs.done and obs.day <= DAYS_PER_EPISODE:
        role = AGENT_ROLES[role_idx % len(AGENT_ROLES)]
        role_idx += 1; turn += 1

        role_data = env._get_role_data(role)
        obs_dict = {
            "agent_id": role, "day": obs.day, "phase": obs.phase,
            "kpis": obs.kpis, "budget_remaining": obs.budget_remaining,
            "recent_actions": obs.recent_actions, "messages": obs.messages,
            "events": obs.events, "role_data": role_data,
            "last_action_result": obs.last_action_result, "done": obs.done, "reward": obs.reward,
        }

        user_msg = agents[role]._build_user_message(obs_dict, turn)
        action_dict = EXPERT_POLICIES[role](obs_dict, turn)

        action = OfficeOsAction(
            agent_id=role, action_type=action_dict["action_type"],
            target=action_dict.get("target", ""),
            parameters=action_dict.get("parameters", {}),
            reasoning=action_dict.get("reasoning", ""),
            message=action_dict.get("message"),
        )
        obs = env.step(action)

        all_records.append({
            "role": role, "system_prompt": ROLE_PROMPTS[role],
            "user_message": user_msg, "assistant_response": action_dict,
            "reward": obs.reward,
        })

    kpis = env._market.get_all_kpis()
    won = len([c for c in env._market.customers if c.stage == "closed_won"])
    print(f"Ep {ep:2d} ({scenario:10s}): Rev=${kpis['total_revenue']:>8,.0f} | Won={won} | Features={kpis['features_shipped']} | Records={len(all_records)}")

# Split by role
role_data = {r: [x for x in all_records if x["role"] == r] for r in AGENT_ROLES}
print(f"\nTotal: {len(all_records)} trajectories")
for role, recs in role_data.items():
    rewards = [r["reward"] for r in recs]
    print(f"  {role:10s}: {len(recs):4d} traj, avg reward: {sum(rewards)/len(rewards):+.2f}")

In [ ]:
# ── Peek at a sample trajectory ─────────────────────────────────
sample = role_data[TRAIN_ROLE][0]
print(f"=== Sample {TRAIN_ROLE} trajectory ===")
print(f"\nUser message (first 500 chars):")
print(sample["user_message"][:500])
print(f"\nExpert response:")
print(json.dumps(sample["assistant_response"], indent=2))
print(f"\nReward: {sample['reward']:.2f}")

## 3. Load Base Model with Unsloth

4-bit QLoRA: fits 7B in ~6GB VRAM, 14B in ~12GB.

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=4096, load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32, use_gradient_checkpointing="unsloth", random_state=3407,
)
print(f"Loaded: {BASE_MODEL} (4-bit QLoRA, rank=32)")
model.print_trainable_parameters()

## 4. GRPO Training — Reward Functions + Trainer

GRPO generates multiple completions per prompt, scores them with reward functions,
and reinforces the better ones. Our format scorer checks:
- Valid JSON (+0.3), clean output (+0.1)
- Correct action_type for role (+0.3, or -0.2 for wrong-role action)
- Reasoning quality (+0.05 to +0.2), target (+0.1), message format (+0.1)

In [ ]:
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

# Build dataset
valid_actions = ROLE_ACTIONS[TRAIN_ROLE]
train_rows = [{
    "prompt": [{"role": "system", "content": t["system_prompt"]},
               {"role": "user", "content": t["user_message"]}],
    "expected_action": json.dumps(t["assistant_response"]),
    "trajectory_reward": t["reward"],
} for t in role_data[TRAIN_ROLE]]

dataset = Dataset.from_list(train_rows)
print(f"Dataset: {len(dataset)} examples for '{TRAIN_ROLE}'")
print(f"Valid actions: {valid_actions}")

In [ ]:
# ── Reward function: format & validity scorer ───────────────────

def score_completion(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else str(completion)
        score = 0.0
        parsed = None
        try:
            start, end = text.find("{"), text.rfind("}") + 1
            if start >= 0 and end > start:
                parsed = json.loads(text[start:end])
                score += 0.3  # Valid JSON
        except (json.JSONDecodeError, ValueError):
            pass
        if parsed:
            before = text[:text.find("{")].strip()
            after = text[text.rfind("}") + 1:].strip()
            if not before and not after: score += 0.1  # Clean output
        if parsed and isinstance(parsed, dict):
            if "action_type" in parsed:
                score += 0.2
                if parsed["action_type"] in valid_actions: score += 0.3
                else: score -= 0.2
            reasoning = parsed.get("reasoning", "")
            if reasoning: score += min(0.2, max(0.05, len(reasoning.split()) * 0.01))
            if parsed.get("target") and len(str(parsed["target"])) > 1: score += 0.1
            msg = parsed.get("message", "")
            if msg and ":" in str(msg): score += 0.1
        rewards.append(max(score, 0.0))
    return rewards

# Quick test
good = ['{"action_type": "BUILD_FEATURE", "target": "SSO", "reasoning": "Building for Acme demo", "message": "sales: building SSO"}']
bad = ['I think we should build something']
print(f"Good output score: {score_completion([good])}")
print(f"Bad output score:  {score_completion([bad])}")

In [ ]:
# ── Train with GRPO ────────────────────────────────────────────
report_to, run_name = "none", None
if WANDB_PROJECT:
    try:
        import wandb
        report_to, run_name = "wandb", f"{TRAIN_ROLE}-colab"
        os.environ.setdefault("WANDB_PROJECT", WANDB_PROJECT)
        print(f"W&B enabled: {WANDB_PROJECT}/{run_name}")
    except ImportError: pass

training_args = GRPOConfig(
    output_dir=f"/tmp/office_os_lora/{TRAIN_ROLE}",
    use_vllm=False,
    num_generations=4,
    max_prompt_length=3072,
    max_completion_length=1024,
    temperature=0.9,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    max_steps=MAX_STEPS,
    bf16=True,
    optim="adamw_8bit",
    logging_steps=1,
    save_steps=999,
    report_to=report_to,
    run_name=run_name,
)

trainer = GRPOTrainer(
    model=model, args=training_args, train_dataset=dataset,
    reward_funcs=[score_completion], tokenizer=tokenizer,
)

print(f"\nTraining '{TRAIN_ROLE}' — {len(dataset)} trajectories, {MAX_STEPS} steps")
print(f"~10-15 min on T4, ~5 min on A100\n")
trainer.train()

In [ ]:
# ── Save LoRA ──────────────────────────────────────────────────
adapter_path = f"/tmp/office_os_lora/{TRAIN_ROLE}/adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"LoRA saved: {adapter_path}")

# Push to HuggingFace
if HF_REPO:
    from huggingface_hub import HfApi
    HfApi().upload_folder(
        folder_path=adapter_path, repo_id=HF_REPO,
        path_in_repo=f"{TRAIN_ROLE}/step-1",
        commit_message=f"LoRA {TRAIN_ROLE} from Colab",
    )
    print(f"Pushed to HF: {HF_REPO}/{TRAIN_ROLE}/step-1")

if WANDB_PROJECT:
    try:
        import wandb
        if wandb.run: wandb.finish()
    except: pass

print("Training complete!")

## 5. Inference — Base Model vs Fine-Tuned

Side-by-side comparison on the same game state.

In [ ]:
import time

# Use a real trajectory from the simulator as the test prompt
test_sample = role_data[TRAIN_ROLE][len(role_data[TRAIN_ROLE]) // 2]  # Mid-game state
test_system = test_sample["system_prompt"]
test_user = test_sample["user_message"]
test_expert = test_sample["assistant_response"]

# Add JSON instructions to system prompt (same as training)
actions_list = "\n".join(f"  - {a}" for a in ROLE_ACTIONS[TRAIN_ROLE])
test_system += (
    f"\n\n## CRITICAL INSTRUCTIONS\nYou can ONLY use these actions:\n{actions_list}\n\n"
    f"Respond with ONLY a valid JSON object, nothing else:\n"
    f'{{"action_type": "<action>", "target": "...", "parameters": {{}}, '
    f'"reasoning": "...", "message": "role: message"}}'
)

def run_inference(mdl, label):
    FastLanguageModel.for_inference(mdl)
    messages = [{"role": "system", "content": test_system}, {"role": "user", "content": test_user}]
    input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(mdl.device)
    t0 = time.time()
    with torch.no_grad():
        outputs = mdl.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    elapsed = time.time() - t0
    text = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*55}")
    print(f"  {label} ({elapsed:.1f}s)")
    print(f"{'='*55}")
    try:
        s, e = text.find("{"), text.rfind("}") + 1
        if s >= 0 and e > s:
            p = json.loads(text[s:e])
            v = 'valid' if p.get('action_type','') in ROLE_ACTIONS[TRAIN_ROLE] else 'INVALID'
            print(f"  Action:    {p.get('action_type','?')} ({v})")
            print(f"  Target:    {p.get('target','')}")
            print(f"  Reasoning: {p.get('reasoning','')}")
            print(f"  Message:   {p.get('message','')}")
        else:
            print(f"  Raw: {text[:300]}")
    except:
        print(f"  Raw: {text[:300]}")
    return text

# Show expert answer first
print(f"{'='*55}")
print(f"  EXPERT ANSWER (ground truth)")
print(f"{'='*55}")
print(f"  Action:    {test_expert['action_type']}")
print(f"  Target:    {test_expert.get('target','')}")
print(f"  Reasoning: {test_expert.get('reasoning','')}")

# Fine-tuned
ft_out = run_inference(model, "FINE-TUNED MODEL")

# Base model (disable LoRA)
model.disable_adapter_layers()
base_out = run_inference(model, "BASE MODEL (no LoRA)")
model.enable_adapter_layers()

# Score comparison
ft_score = score_completion([[{"content": ft_out}]])[0]
base_score = score_completion([[{"content": base_out}]])[0]
print(f"\n{'='*55}")
print(f"  SCORE: Fine-tuned={ft_score:.2f} vs Base={base_score:.2f} (delta={ft_score-base_score:+.2f})")
print(f"{'='*55}")

## 6. Hot-Load LoRA into vLLM (Production)

Push the trained LoRA into a running vLLM server without restart.

In [ ]:
import urllib.request

# Set your vLLM endpoint (Northflank or local)
VLLM_URL = ""  # e.g. "https://vllm--jupyter-pytorch--xxx.code.run" or "http://localhost:8080"

if VLLM_URL:
    lora_name = f"office-os-{TRAIN_ROLE}"
    payload = json.dumps({"lora_name": lora_name, "lora_path": adapter_path}).encode()
    req = urllib.request.Request(
        f"{VLLM_URL}/v1/load_lora_adapter",
        data=payload, method="POST", headers={"Content-Type": "application/json"},
    )
    try:
        with urllib.request.urlopen(req, timeout=30) as resp:
            print(f"LoRA hot-loaded into vLLM: {lora_name}")
            print(f"\nTest it:")
            print(f"  curl {VLLM_URL}/v1/chat/completions \\")
            print(f"    -d '{{\"model\": \"{lora_name}\", \"messages\": [{{\"role\": \"user\", \"content\": \"test\"}}]}}'")
    except Exception as e:
        print(f"Hot-load failed: {e}")
        print("Ensure adapter_path is accessible from the vLLM server.")
        print(f"If using HF, the server can load from: {HF_REPO}/{TRAIN_ROLE}/step-1")
else:
    print("No VLLM_URL set. To hot-load later:")
    print(f'  curl -X POST $VLLM_URL/v1/load_lora_adapter \\')
    print(f'    -d \'{{"lora_name": "office-os-{TRAIN_ROLE}", "lora_path": "{adapter_path}"}}\'')
    print(f"\nLoRA adapter saved at: {adapter_path}")

## 7. Train All 6 Roles (Optional)

Uncomment and run to create a full set of LoRA adapters.
~1-2 hours on T4, ~30 min on A100.

In [ ]:
# import gc
# ALL_ROLES = ["ceo", "dev", "marketing", "sales", "content", "hr"]
# for role in ALL_ROLES:
#     print(f"\n{'#'*60}\n  Training: {role}\n{'#'*60}")
#     del model, trainer; gc.collect(); torch.cuda.empty_cache()
#     model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE_MODEL, max_seq_length=4096, load_in_4bit=True)
#     model = FastLanguageModel.get_peft_model(model, r=32,
#         target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
#         lora_alpha=32, use_gradient_checkpointing="unsloth", random_state=3407)
#     valid_actions = ROLE_ACTIONS[role]
#     rows = [{"prompt": [{"role":"system","content":t["system_prompt"]},{"role":"user","content":t["user_message"]}],
#              "expected_action": json.dumps(t["assistant_response"]), "trajectory_reward": t["reward"]}
#             for t in role_data[role]]
#     ds = Dataset.from_list(rows)
#     args = GRPOConfig(output_dir=f"/tmp/office_os_lora/{role}", use_vllm=False, num_generations=4,
#         max_prompt_length=3072, max_completion_length=1024, temperature=0.9, learning_rate=LEARNING_RATE,
#         per_device_train_batch_size=1, gradient_accumulation_steps=4, num_train_epochs=3,
#         max_steps=MAX_STEPS, bf16=True, optim="adamw_8bit", logging_steps=5, save_steps=999, report_to="none")
#     t = GRPOTrainer(model=model, args=args, train_dataset=ds, reward_funcs=[score_completion], tokenizer=tokenizer)
#     t.train()
#     path = f"/tmp/office_os_lora/{role}/adapter"
#     model.save_pretrained(path); tokenizer.save_pretrained(path)
#     if HF_REPO:
#         from huggingface_hub import HfApi
#         HfApi().upload_folder(folder_path=path, repo_id=HF_REPO, path_in_repo=f"{role}/step-1",
#                               commit_message=f"LoRA {role} from Colab")
#     print(f"  Done: {path}")
# print("\nAll roles trained!")